In [ ]:
%matplotlib inline

# Disable sample shuffling prior to cross-validation

## Problem

Cross-validation shuffles the training samples before splitting,
in order to avoid problems with ordered data.
How can I use the training samples without shuffling?

## Solution

Set the argument `randomize` of the
[compute_cross_validation_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_cross_validation_measure]
method to `False`.

## Step-by-step guide


In [ ]:
from __future__ import annotations

from gemseo import sample_disciplines
from gemseo.algos.doe.openturns.settings.ot_opt_lhs import OT_OPT_LHS_Settings
from gemseo.machine_learning.regression.models.rbf import RBFRegressor
from gemseo.machine_learning.regression.models.rbf_settings import RBFRegressor_Settings
from gemseo.machine_learning.regression.quality.r2_measure import R2Measure
from gemseo.problems.uncertainty.wing_weight.discipline import WingWeightDiscipline
from gemseo.problems.uncertainty.wing_weight.uncertain_space import (
    WingWeightUncertainSpace,
)

### 1. Define the reference model

In this how-to guide,
we consider the wing weight problem
defined in [this page][gemseo.problems.uncertainty.wing_weight].



In [ ]:
discipline = WingWeightDiscipline()
input_space = WingWeightUncertainSpace()

### 2. Create the training dataset

We generate $3 \times d$ training samples
using optimized Latin hypercube sampling strategy.



In [ ]:
training_dataset = sample_disciplines(
    [discipline],
    input_space,
    ["Ww"],
    algo_settings_model=OT_OPT_LHS_Settings(n_samples=input_space.dimension * 3),
)

### 2. Create the ML model

We create a regressor from this training dataset,
taking care to normalize the data to facilitate learning.



In [ ]:
regressor = RBFRegressor(
    training_dataset,
    settings=RBFRegressor_Settings(transformer=RBFRegressor.DEFAULT_TRANSFORMER),
)
regressor.learn()

### 3. Evaluate its quality by resampling



In [ ]:
r2 = R2Measure(regressor)
r2.compute_cross_validation_measure()

Evaluating the quality by cross-validation a second time leads to a new result
because of the randomization of the samples preceding their splitting.



In [ ]:
r2.compute_cross_validation_measure()

Set the `randomize` argument to `False` so as not to shuffle the samples.



In [ ]:
r2.compute_cross_validation_measure(randomize=False)

Let's do it again.



In [ ]:
r2.compute_cross_validation_measure(randomize=False)

## Summary

Cross-validation can avoid shuffling learning samples
by setting the attribute `randomize` of
[compute_cross_validation_measure()][gemseo.machine_learning.core.quality.base_ml_model_quality.BaseMLModelQuality.compute_cross_validation_measure]
to `False`.

